In [2]:
spark.sql("DROP TABLE IF EXISTS fact_sales")
spark.sql("DROP TABLE IF EXISTS dim_category")
spark.sql("DROP TABLE IF EXISTS dim_region")
spark.sql("DROP TABLE IF EXISTS dim_segment")
spark.sql("DROP TABLE IF EXISTS dim_shipmode")
spark.sql("DROP TABLE IF EXISTS dim_date")

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 3, Finished, Available, Finished, False)

DataFrame[]

In [3]:
silver_df = spark.table(
    "`Superstore Data Engineering`.SuperstoreLakehouse.dbo.silver_superstore"
)

display(silver_df)

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8f7ac71c-49de-419d-9211-8d8b96587825)

In [4]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 5, Finished, Available, Finished, False)

In [5]:
window = Window.orderBy("category")

dim_category = (
    silver_df
    .select("category")
    .distinct()
    .withColumn(
        "category_key",
        row_number().over(window)
    )
)

display(dim_category)

dim_category.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("dim_category")

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c73715d4-e487-4094-8084-c60c086b3d4c)

In [6]:
window = Window.orderBy(
    "country",
    "region",
    "state",
    "city",
    "postal_code"
)

dim_region = (
    silver_df
    .select(
        "country",
        "region",
        "state",
        "city",
        "postal_code"
    )
    .distinct()
    .withColumn(
        "region_key",
        row_number().over(window)
    )
)

display(dim_region)

dim_region.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("dim_region")

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 57d95d58-9d28-4913-9ed3-49f8efd70e6a)

In [7]:
window = Window.orderBy("segment")

dim_segment = (
    silver_df
    .select("segment")
    .distinct()
    .withColumn(
        "segment_key",
        row_number().over(window)
    )
)

display(dim_segment)

dim_segment.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("dim_segment")

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77a1fb53-5d2a-4792-9544-bf74a1339411)

In [8]:
window = Window.orderBy("ship_mode")

dim_shipmode = (
    silver_df
    .select("ship_mode")
    .distinct()
    .withColumn(
        "shipmode_key",
        row_number().over(window)
    )
)

display(dim_shipmode)

dim_shipmode.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("dim_shipmode")

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0293c11d-726c-459e-8b13-07d222a2cb66)

In [9]:
from pyspark.sql import Row

dates = [
    Row(
        date_key=1,
        year=2024,
        quarter=1,
        month=1,
        month_name="January"
    )
]

dim_date = spark.createDataFrame(dates)

display(dim_date)

dim_date.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("dim_date")

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3e7e351d-dc6c-449e-86f5-28368842ddd2)

In [10]:
fact_sales = (
    silver_df
    .join(dim_category, "category", "left")
    .join(
        dim_region,
        ["country", "region", "state", "city", "postal_code"],
        "left"
    )
    .join(dim_segment, "segment", "left")
    .join(dim_shipmode, "ship_mode", "left")
)

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 11, Finished, Available, Finished, False)

In [11]:
print("Fact rows:", fact_sales.count())

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 12, Finished, Available, Finished, False)

Fact rows: 9977


In [12]:
fact_sales = fact_sales.select(
    "category_key",
    "region_key",
    "segment_key",
    "shipmode_key",
    "sales",
    "quantity",
    "discount",
    "profit"
)

display(fact_sales)

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f03d0f86-320a-4ac2-8944-52075b0049a6)

In [13]:
print("Fact rows:", fact_sales.count())

display(fact_sales)

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 14, Finished, Available, Finished, False)

Fact rows: 9977


SynapseWidget(Synapse.DataFrame, 23cd0067-675b-4833-803d-e61d371bb1b4)

In [14]:
fact_sales.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("fact_sales")

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 15, Finished, Available, Finished, False)

In [15]:
display(spark.table("dim_category"))
display(spark.table("dim_region"))
display(spark.table("dim_segment"))
display(spark.table("dim_shipmode"))
display(spark.table("dim_date"))
display(spark.table("fact_sales"))

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8a8ca0e0-0f21-48c2-aedc-594e08f743e3)

SynapseWidget(Synapse.DataFrame, f5dabb2f-667e-48c5-b29c-5dbd684c1bc8)

SynapseWidget(Synapse.DataFrame, ae068808-3a8b-47ff-86b1-793523247051)

SynapseWidget(Synapse.DataFrame, 1b866d29-8dbb-486f-990f-1cb529ef0378)

SynapseWidget(Synapse.DataFrame, 9a939e63-118e-432a-a4e4-9d3178194418)

SynapseWidget(Synapse.DataFrame, 74fd4c8b-816a-4d90-ab9a-8dd677f05cdd)

In [16]:
print("Category:", dim_category.count())
print("Region:", dim_region.count())
print("Segment:", dim_segment.count())
print("Ship Mode:", dim_shipmode.count())
print("Date:", dim_date.count())
print("Fact:", fact_sales.count())

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 17, Finished, Available, Finished, False)

Category: 3
Region: 632
Segment: 3
Ship Mode: 4
Date: 1
Fact: 9977


In [17]:
spark.sql("SHOW TABLES").show(truncate=False)

print("Bronze:", spark.table("bronze_superstore").count())
print("Silver:", spark.table("silver_superstore").count())
print("Fact:", spark.table("fact_sales").count())

StatementMeta(, 37b2103f-301e-4ee1-8b7b-834a4e923964, 18, Finished, Available, Finished, False)

+-----------------------------------------------------+-----------------+-----------+
|namespace                                            |tableName        |isTemporary|
+-----------------------------------------------------+-----------------+-----------+
|`Superstore Data Engineering`.SuperstoreLakehouse.dbo|bronze_superstore|false      |
|`Superstore Data Engineering`.SuperstoreLakehouse.dbo|dim_category     |false      |
|`Superstore Data Engineering`.SuperstoreLakehouse.dbo|dim_date         |false      |
|`Superstore Data Engineering`.SuperstoreLakehouse.dbo|dim_region       |false      |
|`Superstore Data Engineering`.SuperstoreLakehouse.dbo|dim_segment      |false      |
|`Superstore Data Engineering`.SuperstoreLakehouse.dbo|dim_shipmode     |false      |
|`Superstore Data Engineering`.SuperstoreLakehouse.dbo|fact_sales       |false      |
|`Superstore Data Engineering`.SuperstoreLakehouse.dbo|silver_superstore|false      |
+-----------------------------------------------------